# Preprocessing — Jalur **SVM + TF-IDF**  (MongoDB Atlas)

Self-contained (tanpa `import src`). Baca `raw_comments` (`in_balanced_set=true`),
preprocessing agresif, tulis ke **`processed_svm`**.

**Kenapa agresif?** TF-IDF = *bag-of-words* — tak paham konteks/urutan, jadi kita
maksimalkan rasio sinyal:
1. **clean_for_svm** — lowercase; buang URL/mention/hashtag/emoji/angka/tanda baca.
2. **normalize_slang** — `gak→tidak`, `sdh→sudah`, … (satukan varian).
3. **remove_stopword** — buang kata fungsi (noise). **Negasi (tidak/bukan/belum)
   SENGAJA dipertahankan** — krusial utk sentimen.
4. **stemming (Sastrawi)** — `kebohongan→bohong` (kurangi sparsity).

> Split identik dgn IndoBERT (urut `comment_id` + seed=42, sebelum preprocessing).

## 0. Dependency

In [1]:
%pip install -q "pymongo[srv]" dnspython certifi python-dotenv PySastrawi scikit-learn pandas numpy

Note: you may need to restart the kernel to use updated packages.


## 1. Baca dataset balanced dari MongoDB

In [2]:
import os, pandas as pd
from pymongo import MongoClient
import certifi

MONGO_URI = os.environ.get("MONGO_URI", "")
if not MONGO_URI:
    try:
        from dotenv import load_dotenv; load_dotenv(); MONGO_URI = os.environ.get("MONGO_URI","")
    except Exception: pass
if not MONGO_URI:
    from getpass import getpass; MONGO_URI = getpass("MONGO_URI: ")
DB_NAME = os.environ.get("MONGO_DB_NAME","youtube_sentiment")
client = MongoClient(MONGO_URI, tlsCAFile=certifi.where(), serverSelectionTimeoutMS=20000)
client.admin.command("ping"); print("Koneksi MongoDB OK.")
LABELS = ["Negatif","Netral","Positif"]
docs = list(client[DB_NAME]["raw_comments"].find({"in_balanced_set":True},
            {"_id":0,"comment_id":1,"video_id":1,"text":1,"label":1}))
df = pd.DataFrame(docs); df = df[df["label"].isin(LABELS)].copy()
print(f"{len(df)} dok balanced dari Mongo"); print(df["label"].value_counts().reindex(LABELS).to_string())

Koneksi MongoDB OK.


3000 dok balanced dari Mongo
label
Negatif    1000
Netral     1000
Positif    1000


## 2. Fungsi preprocessing (inline, identik dgn src/text_normalizer.py)

In [3]:
import re
from typing import List

SLANG_DICT = {'aja': 'saja',
 'bagus': 'bagus',
 'baguss': 'bagus',
 'bang': '',
 'banget': 'sangat',
 'bener': 'benar',
 'bgt': 'sangat',
 'blm': 'belum',
 'blom': 'belum',
 'bngt': 'sangat',
 'bro': '',
 'buat': 'untuk',
 'bwt': 'untuk',
 'channel': 'saluran',
 'cuma': 'hanya',
 'dah': 'sudah',
 'deh': '',
 'dg': 'dengan',
 'dgn': 'dengan',
 'dong': '',
 'dr': 'dari',
 'dri': 'dari',
 'elu': 'kamu',
 'enggak': 'tidak',
 'g': 'tidak',
 'ga': 'tidak',
 'gak': 'tidak',
 'gakk': 'tidak',
 'gan': '',
 'gila': 'gila',
 'gilaa': 'gila',
 'gk': 'tidak',
 'gua': 'saya',
 'gue': 'saya',
 'gw': 'saya',
 'haha': '',
 'hehe': '',
 'hihi': '',
 'iya': 'ya',
 'iyaa': 'ya',
 'jelek': 'jelek',
 'jelekk': 'jelek',
 'jg': 'juga',
 'jga': 'juga',
 'jgn': 'jangan',
 'jkw': 'jokowi',
 'kak': '',
 'kalo': 'kalau',
 'kalu': 'kalau',
 'kan': '',
 'karna': 'karena',
 'keren': 'keren',
 'kerennn': 'keren',
 'kl': 'kalau',
 'klo': 'kalau',
 'klu': 'kalau',
 'kok': '',
 'komen': 'komentar',
 'konten': 'konten',
 'krn': 'karena',
 'kwkw': '',
 'lah': '',
 'lg': 'lagi',
 'lgi': 'lagi',
 'like': 'suka',
 'lo': 'kamu',
 'lu': 'kamu',
 'makasi': 'terima kasih',
 'makasih': 'terima kasih',
 'mantap': 'mantap',
 'mantep': 'mantap',
 'min': '',
 'mksh': 'terima kasih',
 'ngga': 'tidak',
 'nggak': 'tidak',
 'nih': '',
 'ntar': 'nanti',
 'ok': 'oke',
 'org': 'orang',
 'q': 'saya',
 'roi': 'roy',
 'sdh': 'sudah',
 'share': 'bagikan',
 'sih': '',
 'smgt': 'semangat',
 'sub': 'berlangganan',
 'subscribe': 'berlangganan',
 'sy': 'saya',
 'tak': 'tidak',
 'tar': 'nanti',
 'tau': 'tahu',
 'tdk': 'tidak',
 'tks': 'terima kasih',
 'tp': 'tapi',
 'tpi': 'tapi',
 'trs': 'terus',
 'trus': 'terus',
 'udah': 'sudah',
 'upload': 'unggah',
 'utk': 'untuk',
 'wkwk': '',
 'wkwkwk': '',
 'xD': '',
 'xd': '',
 'yg': 'yang',
 'yng': 'yang'}

STOPWORDS_ID = {'ada',
 'adalah',
 'agar',
 'akan',
 'amat',
 'anda',
 'antara',
 'apa',
 'atas',
 'atau',
 'bagaimana',
 'baru',
 'bawah',
 'beberapa',
 'besar',
 'bisa',
 'dalam',
 'dan',
 'dari',
 'dengan',
 'di',
 'dia',
 'dua',
 'ia',
 'ini',
 'itu',
 'jarang',
 'jika',
 'juga',
 'kadang',
 'kami',
 'kamu',
 'karena',
 'ke',
 'kecil',
 'ketika',
 'kita',
 'lagi',
 'lain',
 'lama',
 'lebih',
 'luar',
 'maka',
 'masih',
 'mengapa',
 'mereka',
 'namun',
 'nya',
 'oleh',
 'pada',
 'pernah',
 'pun',
 'saat',
 'saja',
 'sama',
 'sangat',
 'satu',
 'saya',
 'sebelum',
 'sedang',
 'sejak',
 'sekali',
 'selalu',
 'selama',
 'semua',
 'sering',
 'setelah',
 'setiap',
 'si',
 'sudah',
 'supaya',
 'tapi',
 'telah',
 'tetapi',
 'tiga',
 'untuk',
 'ya',
 'yang'}

_RE_URL = re.compile('https?://\\S+|www\\.\\S+', re.IGNORECASE)
_RE_MENTION = re.compile('@\\w+')
_RE_HASHTAG = re.compile('#\\w+')
_RE_EMOJI = re.compile('[😀-🙏🌀-🗿🚀-\U0001f6ff🜀-🝿🞀-\U0001f7ff🠀-\U0001f8ff🤀-🧿🨀-\U0001fa6f🩰-\U0001faff✂-➰Ⓜ-🉑🤦-🤷\u200d♀-♂☀-⭕⏏⏩⌚️〰]+', re.UNICODE)
_RE_NUMBER = re.compile('\\b\\d+\\b')
_RE_NON_ALPHA = re.compile('[^a-z\\s]')
_RE_MULTI_SPACE = re.compile('\\s+')

def clean_for_svm(text: str) -> str:
    """Cleaning agresif untuk jalur SVM + TF-IDF."""
    if not text or not isinstance(text, str):
        return ""
    t = text.lower()
    t = _RE_URL.sub(" ", t)
    t = _RE_MENTION.sub(" ", t)
    t = _RE_HASHTAG.sub(" ", t)
    t = _RE_EMOJI.sub(" ", t)
    t = _RE_NUMBER.sub(" ", t)
    t = _RE_NON_ALPHA.sub(" ", t)
    t = _RE_MULTI_SPACE.sub(" ", t)
    return t.strip()


def normalize_slang(text: str, slang_dict: dict = SLANG_DICT) -> str:
    """Ganti kata slang/informal dengan bentuk baku."""
    if not text:
        return ""
    tokens = text.split()
    normalized = [slang_dict.get(tok, tok) for tok in tokens]
    result = " ".join(tok for tok in normalized if tok)
    return _RE_MULTI_SPACE.sub(" ", result).strip()


def tokenize(text: str) -> List[str]:
    """Tokenisasi sederhana berdasarkan whitespace."""
    return text.split() if text else []


def remove_stopwords(tokens: List[str], stopwords: set = STOPWORDS_ID) -> List[str]:
    """Hapus stopword dari daftar token."""
    return [tok for tok in tokens if tok not in stopwords and len(tok) > 1]


def preprocess_svm_python(text: str) -> str:
    """Pipeline SVM tanpa stemming (stemming dilakukan terpisah via Sastrawi UDF)."""
    cleaned = clean_for_svm(text)
    normalized = normalize_slang(cleaned)
    tokens = tokenize(normalized)
    tokens = remove_stopwords(tokens)
    return " ".join(tokens)

In [4]:
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
_stemmer = StemmerFactory().create_stemmer()
def make_text_svm(text: str) -> str:
    pre = preprocess_svm_python(text or "")
    return _stemmer.stem(pre) if pre else pre

## 3. Transformasi langkah-demi-langkah

In [5]:
def trace_svm(t):
    s1=clean_for_svm(t); s2=normalize_slang(s1); s3=" ".join(remove_stopwords(tokenize(s2))); s4=_stemmer.stem(s3)
    print(f"RAW             : {t!r}")
    print(f"1. clean_for_svm  : {s1!r}")
    print(f"2. normalize_slang: {s2!r}")
    print(f"3. remove_stopword: {s3!r}")
    print(f"4. stem (FINAL)   : {s4!r}")
trace_svm("Ijazahnya tidak palsu kok, jgn asal tuduh tanpa bukti")

RAW             : 'Ijazahnya tidak palsu kok, jgn asal tuduh tanpa bukti'
1. clean_for_svm  : 'ijazahnya tidak palsu kok jgn asal tuduh tanpa bukti'
2. normalize_slang: 'ijazahnya tidak palsu jangan asal tuduh tanpa bukti'
3. remove_stopword: 'ijazahnya tidak palsu jangan asal tuduh tanpa bukti'
4. stem (FINAL)   : 'ijazah tidak palsu jangan asal tuduh tanpa bukti'


### Tabel before→after (sampel acak)

In [6]:
import pandas as pd
pd.set_option("display.max_colwidth", 55)
_s = df.sample(6, random_state=7).copy()
_s["svm"] = _s["text"].astype(str).map(make_text_svm)
_s[["label","text","svm"]]

,label,text,svm
1306,Negatif,Keren cadas RH,keren cadas rh
2037,Negatif,Orang2 aneh.kata2 maaf aja di ributin d permasalahk...,orang aneh kata maaf ributin masalah aneh
568,Negatif,Kandangin seumur hidup... biar Indonesia ngak gaduh...,kandangin umur hidup biar indonesia ngak gaduh
1897,Positif,Kalau nonton mas Roy Suryo aku senyum2 sendiri.\nSe...,kalau nonton mas roy suryo aku senyum sendiri moga ...
2498,Negatif,Bu Diana mantap. IQ nya tinggi sekali.,bu ana mantap iq tinggi
1111,Positif,Inget yang asli yang ada badaknya 🦏🦏🦏🦏🦏🦏🦏🦏🦏🦏🦏🦏🦏🦏🦏🦏,inget asli badak


## 4. Terapkan ke semua data + split identik (70/20/10)

In [7]:
from sklearn.model_selection import train_test_split
LABEL2ID={l:i for i,l in enumerate(LABELS)}
df = df.sort_values("comment_id").reset_index(drop=True)
df["label_id"]=df["label"].map(LABEL2ID)
tmp, te = train_test_split(df, test_size=0.10, stratify=df["label_id"], random_state=42)
tr, va = train_test_split(tmp, test_size=0.20/0.90, stratify=tmp["label_id"], random_state=42)
splits={}
for nm,part in [("train",tr),("val",va),("test",te)]:
    part=part.copy(); part["svm"]=part["text"].astype(str).map(make_text_svm)
    b=len(part); part=part[part["svm"].str.len()>0]
    if b-len(part): print(f"[{nm}] {b-len(part)} dibuang (kosong)")
    part["split"]=nm
    splits[nm]=part.reset_index(drop=True)
for nm,part in splits.items():
    print(f"[{nm}] n={len(part)} | "+" ".join(f"{k}={v}" for k,v in part['label'].value_counts().reindex(LABELS).items()))

[train] 2 dibuang (kosong)


[train] n=2098 | Negatif=699 Netral=699 Positif=700
[val] n=600 | Negatif=200 Netral=200 Positif=200
[test] n=300 | Negatif=100 Netral=100 Positif=100


## 5. EDA & ringkasan kualitas

In [8]:
# === EDA & ringkasan kualitas ===
import numpy as np
proc = pd.concat(splits.values(), ignore_index=True)
print("Distribusi kelas (after preprocessing):")
print(proc["label"].value_counts().reindex(LABELS).to_string())

wl_o = proc["text"].astype(str).str.split().map(len)
wl_p = proc["svm"].astype(str).str.split().map(len)
print(f"\nPanjang (kata)  ORIG : mean {wl_o.mean():.1f} | median {wl_o.median():.0f} | p95 {wl_o.quantile(.95):.0f} | max {wl_o.max()}")
print(f"Panjang (kata)  SVM  : mean {wl_p.mean():.1f} | median {wl_p.median():.0f} | p95 {wl_p.quantile(.95):.0f} | max {wl_p.max()}")

vocab = set(t for s in proc["svm"].astype(str) for t in s.split())
dup = proc["svm"].duplicated().sum()
empty = (wl_p<=1).sum()
cross = (proc.groupby("svm")["split"].nunique()>1).sum()
print(f"\nVocab unik       : {len(vocab)}")
print(f"Baris <=1 kata    : {empty} ({empty/len(proc)*100:.1f}%)")
print(f"Duplikat teks     : {dup} ({dup/len(proc)*100:.1f}%)")
print(f"Teks lintas split (LEAKAGE): {cross}  -> harus 0")

Distribusi kelas (after preprocessing):
label
Negatif     999
Netral      999
Positif    1000

Panjang (kata)  ORIG : mean 15.6 | median 11 | p95 44 | max 224
Panjang (kata)  SVM  : mean 11.7 | median 9 | p95 31 | max 179

Vocab unik       : 5803
Baris <=1 kata    : 12 (0.4%)
Duplikat teks     : 3 (0.1%)
Teks lintas split (LEAKAGE): 0  -> harus 0


## 6. Top term diskriminatif per kelas

In [9]:
# === Top term diskriminatif per kelas (over-representasi vs global) ===
from collections import Counter
cnt = {l: Counter() for l in LABELS}; tot = Counter(); nclass = {}
for l in LABELS:
    sub = proc[proc["label"]==l]; nclass[l]=len(sub)
    for s in sub["svm"].astype(str):
        for t in set(s.split()): cnt[l][t]+=1; tot[t]+=1
N=len(proc)
def top(l,k=12,minc=5):
    out=[]
    for t,c in cnt[l].items():
        if tot[t]<minc: continue
        lift=(c/nclass[l])/(tot[t]/N)   # >1 = lebih sering di kelas ini drpd rata2
        out.append((t,round(lift,2),c))
    return sorted(out,key=lambda x:-x[1])[:k]
for l in LABELS:
    print(f"[{l}] " + ", ".join(f"{t}({lift})" for t,lift,c in top(l)))

[Negatif] hujat(3.0), pel(3.0), dikit(3.0), stress(3.0), siroy(3.0), daripada(3.0), panci(2.76), tompel(2.69), cengengesan(2.67), gila(2.64), gerombol(2.63), potong(2.63)
[Netral] host(3.0), kpk(3.0), harta(3.0), banjir(3.0), vs(2.67), tidur(2.63), penasaran(2.57), lengkap(2.5), bencana(2.46), istilah(2.4), sinetron(2.4), imbang(2.4)
[Positif] sembunyi(3.0), poto(3.0), pramuka(3.0), janggal(3.0), dian(3.0), curiga(3.0), cetak(3.0), daftar(3.0), bongkar(2.86), kampus(2.66), tunjukin(2.62), podcast(2.57)


## 7. Tulis ke koleksi `processed_svm`

In [10]:
from pymongo import UpdateOne
OUT=client[DB_NAME]["processed_svm"]; ops=[]
for nm,part in splits.items():
    for r in part.to_dict("records"):
        ops.append(UpdateOne({"comment_id":r["comment_id"]},{"$set":{
            "comment_id":r["comment_id"],"video_id":r.get("video_id"),"text":r["text"],
            "svm":r["svm"],"label":r["label"],"label_id":int(r["label_id"]),"split":nm}},upsert=True))
OUT.bulk_write(ops,ordered=False)
print("processed_svm:",OUT.count_documents({}),"dok")

processed_svm: 2998 dok


Selesai. **Lanjut:** `train_svm.ipynb`.